# CIS 536/736: Computer Graphics / Game Engines
## MP 5 Starter Notebook: Geometry Nodes Procedural Asset

**Module 5: Geometry Nodes Fundamentals & Patterns (Lectures 14–16)**  
**Target Environment:** Blender 4.2 LTS / Python 3.10+ (pxr)  
**Scope:** Lab 3a only — single-asset export and validation. Multi-variant composition and Unity import are a later assignment.

---
### Lab Objective
Construct a non-destructive, parameter-driven asset in Blender 4.2 LTS, then use this notebook to verify that your OpenUSD export actually contains valid, realized geometry — not empty transforms from a failed modifier evaluation.

**TODO before you touch this notebook:** complete your Geometry Nodes graph in Blender and export your asset to `.usd`/`.usda`. This notebook validates your export; it does not build your graph for you.

### 1. Base Asset Foundation (Blender — do this first, outside this notebook)
1. Open Blender 4.2 LTS. Delete the default cube; add a base primitive (Plane, Cylinder, or Icosphere).
2. Open the **Geometry Nodes** workspace and attach a new modifier tree.
3. Build your macro-shape with structural nodes (`Transform`, `Extrude Mesh`, `Subdivision Surface`).

### 2. Point Distribution & Instancing (Blender)
* **Scatter:** `Distribute Points on Faces`, mode = **Poisson Disk**.
* **Instance:** `Instance on Points`, fed by an `Object Info` / `Collection Info` node pointing at your detail object(s).
* **Variation:** `Random Value` nodes mapped to the `Scale` and `Rotation` inputs of your instancer, so instances aren't visually identical.

### 3. Exposing Parameters (The Art-Directable API)
Route these from internal node sockets to the blank sockets on your `Group Input` node — they must appear as usable controls in the Modifier Properties panel, **not** stay as hardcoded values:

1. **Seed** (Integer) — cycles random variations.
2. **Density** (Float) — controls your Poisson Disk maximum density.
3. **Scale Variation** (Vector/Float) — min/max bounds for instance randomization.

**Checkpoint:** scrub each of these three sliders in the Modifier panel right now. If nothing visibly changes, fix that before moving on — a parameter that doesn't do anything isn't "exposed," it's decorative.

### 4. Finalizing the Mesh & Material (Blender)
* **`Set Material`:** immediately before `Group Output`. A basic Principled BSDF slot is enough — you are not authoring the final shader here.
* **`Realize Instances`:** at the end of your instancing chain, before any join with base geometry. **This is the most common point of failure in this assignment.** OpenUSD cannot serialize Blender's un-realized instance references — skip this node and your export will contain the base shape but none of your scattered detail.

### 5. Stress-Test Before You Export (Blender)
Scrub Seed, Density, and Scale across their full range and check:
* **Clipping:** do instances intersect unnaturally? (Adjust Poisson Disk `Distance Min`.)
* **Orphans:** do instances float in mid-air? (Check normal alignment / origin points.)
* **Performance:** does viewport framerate collapse with `Realize Instances` active?

Note what you find — this goes in your "Stress-Test Notes" report section.

### 6. Export to OpenUSD (Blender)
1. Apply object scale: `Ctrl+A → Scale`.
2. `File → Export → Universal Scene Description (.usd / .usda)`.
3. Check **Export UVs** and **Export Normals**; set **Evaluation Mode** to `Render` or `Viewport` so the modifier stack output is actually captured.
4. Name the file something structural, e.g. `PROP_ProceduralAsset_v1.usda`, and place it in an `exports/` folder next to this notebook.

### 7. Pipeline Validation: Python USD Stage Check

**YOUR TASK:** complete the `validate_procedural_asset` function below. It must:
1. Confirm the file exists.
2. Open the stage with `Usd.Stage.Open`.
3. Traverse every prim, and for each `UsdGeom.Mesh` prim, print its path and vertex count (via `GetPointsAttr().Get()`).
4. Report **FAIL** if zero mesh prims are found (this is the signature of a missing `Realize Instances` node), and **PASS** otherwise, printing the total mesh count and total vertex count across all meshes.

The scaffold below gives you the imports and function signature. Fill in the `# TODO` blocks — do not just paste the instructor's reference implementation if you've seen one; you should be able to explain every line.

In [ ]:
# Install the USD core library if missing: !pip install usd-core
from pxr import Usd, UsdGeom
import os

def validate_procedural_asset(file_path: str) -> bool:
    """
    Validate that an exported OpenUSD procedural asset contains real,
    realized geometry rather than empty transforms.

    Returns True if at least one UsdGeom.Mesh prim with a nonzero point
    count is found; False otherwise.
    """
    # TODO: check the file exists at `file_path`. If not, print an
    # [ERROR] message and return False.

    # TODO: open the stage with Usd.Stage.Open(file_path). If it fails
    # to open, print an [ERROR] message and return False.

    mesh_count = 0
    total_vertices = 0

    # TODO: traverse the stage (stage.Traverse()). For every prim that
    # IsA(UsdGeom.Mesh):
    #   - increment mesh_count
    #   - get its points via UsdGeom.Mesh(prim).GetPointsAttr().Get()
    #   - add len(points) to total_vertices (watch out for points being None)
    #   - print a [PASS] line with the prim's path and its vertex count

    # TODO: if mesh_count == 0, print a [FAIL] message explaining that
    # this usually means 'Realize Instances' was missing or didn't run,
    # and return False.

    # TODO: otherwise print a [SUCCESS] summary with mesh_count and
    # total_vertices, and return True.

    pass  # remove this once your implementation is complete


# Point this at your own export once it exists:
usd_file_path = "exports/PROP_ProceduralAsset_v1.usda"

# Uncomment to run once validate_procedural_asset is implemented:
# validate_procedural_asset(usd_file_path)

### 8. Final Submission Checklist & GitOps Protocol
- [ ] Geometry Nodes graph uses fields and parameters — no destructive, hardcoded-only modifiers.
- [ ] Seed, Density, and Scale Variation are exposed to the Modifier Panel and functionally verified (Section 3 checkpoint).
- [ ] `Realize Instances` is implemented prior to output; `Set Material` is set before `Group Output`.
- [ ] Stress-test notes recorded (Section 5).
- [ ] Asset exports cleanly to `.usd`/`.usda` without errors.
- [ ] `validate_procedural_asset` is fully implemented (no remaining `# TODO`s) and reports **PASS** against your own export, with output cells visible (do not clear outputs before submitting).
- [ ] The `.blend` source, the `.usd`/`.usda` export, and this completed notebook are committed to your `cis536_736_hw` GitHub repository.
- [ ] The absolute commit URL (not just the repo URL) is posted to the Canvas MP 5 assignment drop, per the course GitOps protocol.